# Fuel Burn Calculations

The mass of the fuel consumed throughout the flight, and possibly consumed during the reserve mission as well, is calculated in aviary_group.py.
The fuel mass can be estimated in premission by assuming that the difference between the gross mass and the zero fuel mass is given to fuel.
However, for off-design missions, this estimate will be inaccurate. For many design missions it can also be inaccurate.
These issues can be compounded by the fact that the aircraft could have some payload it was dropping during the mission. 
Deployable payloads (payloads that are not delivered to the final destination) currently do not have any place in the overall aircraft mass buildup.

![Fuel Breakdown](images/Fuel_Breakdown.png "Details on the Fuel Brakdown is given in the figure.")


## Fuel Mass
The Fuel Mass breakdown is shown below. Note that UNUSABLE_FUEL_MASS is not accounted for in this mass and is instead book kept under USEFUL_LOAD which contributes to OPERATING_MASS see [mass subsystem for details](mass.ipynb).

- TOTAL_FUEL (mass)
    - FUEL (mass)
        - FUEL_TAXI_OUT (mass)
        - Takeoff.FUEL (mass)
        - Fuel burned in regular mission phases (mass) (i.e. anything drescribed in mission info).
    - TOTAL_RESERVE_FUEL (mass)
        - RESERVE_FUEL (mass)
        - RESERVE_FUEL_ADDITIONAL (mass)
        - RESERVE_FUEL_MARGIN (%)

## Maximum Fuel Mass 
The total mass fuel that can fit into the aircraft, regardless of fuel mass or gross mass constraints is as follows:
- TOTAL_CAPACITY (mass)
    - WING_FUEL_CAPACITY (mass)
    - FUSEALGE_FUEL_CAPACITY (mass)
    - AUXILIARY_FUEL_CAPACITY (mass)
    - UNUSABLE_FUEL_MASS (mass)

## Block Fuel
The amount of fuel it takes to get from the departure gate at the source airport to the departure gate at the destination airport, not including any of the reserve mission fuel.
- BLOCK_FUEL (mass)
    - FUEL_TAXI_OUT
    - Takeoff.FUEL
    - Fuel burned in regular mission phases (mass) (i.e. anything drescribed in mission info).
    - FUEL_TAXI_IN

## Caveats
FUEL_TAXI_OUT and Takeoff.FUEL are only applicable to energy-state EOM and should never be used with 2DOF EOMs. 
This is because the latter includes both taxi and takeoff in the regular mission phases.

FUEL_TAXI_IN can be used with either energy-state or 2DOF EOMs, as long as no taxi-in phase has been added in the mission info. 
If a taxi phase is added into the mission info, then that taxi-in fuel burn will end up getting counted twice, once as part of fuel burned in regular mission phases, and once as FUEL_TAXI_IN.
Additionally, FUEL_TAXI_IN is drawn from TOTAL_RESERVE_FUEL. 
There is no constraint enforced to ensure that you have enough reserve fuel to taxi-in.
It is assumed that if you must burn all of your reserve fuel for the reserve mission, then the fact that the aircraft has no fuel leftover to taxi-in to the gate is not an issue.

## Constraints
A fuel mass constraint is used to ensure the fuel mass does not exceed the total fuel allowed in the fuel tanks.
This is essentially a volume constraint but we track it as a mass because all the fuel-tank capacities are tracked as mass not volume.
This is enforced by evaluating {glue:md}`Mission.Constraints.EXCESS_FUEL_CAPACITY`. 
If this value is below zero, then there is more fuel mass than capacity in the tanks. 
This constraint can be disabled by setting {glue:md}`Aircraft.Fuel.IGNORE_FUEL_CAPACITY_CONSTRAINT` to True.

A second mass constraint is used to ensure that the fuel loaded into the aircraft is equal to the fuel used for the main and reserve mission.
This constraint is alaways enforced by {glue:md}`Mission.Constraints.MASS_RESIDUAL`.
If this value goes below zero, that means the mission and reserve mission require more fuel that was originally loaded onto the aircraft.
In standard usage, Aviary forces this constraint to equal zero. 
However, if the user wishes to fly with more fuel than required in order to "resupply fuel to a remote location", the user can set the constraint to lower=0.0 in `aviary_group.py`.
Caution will need to be taken to ensure the ref is of the right magnitude otherwise the optimizer may not try as hard as needed to minimize this value.

## Conversions from GASP to Aviary Fuel
Caution should be taken when converting GASP aircraft and evaluating fuel mass.
FRESF, in GASP can be assigned to three differen values: an extra fuel weight in lbs, an extra fuel percentage, or an extra reserve flight time.
FRESF values between 0 and 10 will not evaluate to a extra reserve flight time. Instead this must be accomplished by setting up a reserve mission and a `target_time`. Doing this will help Aviary calculate {glue:md}`Mission.TOTAL_RESERVE_FUEL` which is the fuel burn for all of the reserve missions. [Read more about generating reserve missions.](../reserve_missions.ipynb)
FRESF values at or below 0 will convert to a percentage of mission fuel burn reserive and bookept under {glue:md}`Mission.RESERVE_FUEL_MARGIN`. (`FRESHF = -.1` -> `RESERVE_FUEL_MARGIN = 10`)
FRESF values at or above 10 will convert directly to lbm fuel burn reserves and accounted for in {glue:md}`Mission.RESERVE_FUEL_ADDITIONAL`. 
Further details on these conversiosn can be viewed in `fortran_to_aviary.py`.

In [ ]:
# Testing Cell
from aviary.api import Aircraft, Mission
from aviary.utils.doctape import get_variable_name, glue_variable

glue_variable(get_variable_name(Mission.Constraints.EXCESS_FUEL_CAPACITY), md_code=True)
glue_variable(get_variable_name(Aircraft.Fuel.IGNORE_FUEL_CAPACITY_CONSTRAINT), md_code=True)
glue_variable(get_variable_name(Mission.Constraints.MASS_RESIDUAL), md_code=True)
glue_variable(get_variable_name(Mission.RESERVE_FUEL_MARGIN), md_code=True)
glue_variable(get_variable_name(Mission.RESERVE_FUEL_ADDITIONAL), md_code=True)
glue_variable(get_variable_name(Mission.RESERVE_FUEL), md_code=True)
glue_variable(get_variable_name(Mission.TOTAL_RESERVE_FUEL), md_code=True)